## Deep Agents
`create_agent` gives you a tool-calling loop. For long, multi-step tasks (research, writing a report, coding) an agent also needs a place to keep notes, a way to delegate sub-tasks and protection against an overflowing context. **Deep Agents** (`deepagents`) is a ready-made agent harness built on LangChain and LangGraph that adds exactly that:
- **virtual file system** – tools `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`; files live in the agent state,
- **subagents** – the `task` tool delegates work to specialised agents with their own prompt and tools, so the main context stays small,
- **context management** – long conversations are summarised automatically (`SummarizationMiddleware`).

The result of `create_deep_agent` is a regular compiled LangGraph graph, so `invoke`, `stream`, checkpointers and LangSmith work as usual.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/en/13_1_Bonus_Deep_agents_en.ipynb)

| | |
|---|---|
| 🎯 **What you'll learn** | Deep Agents: an agent harness with a virtual file system, subagents and context management, built on LangGraph. |
| 🔑 **API keys** | `OPENAI_API_KEY`, `TAVILY_API_KEY` |
| 💲 **API cost** | low – a dozen or so requests |
| ⏱️ **Run time** | ~2 min |

### Install libraries

In [1]:
# Install this notebook's direct dependencies.
# Skipped when pip is missing, e.g. in the `uv sync` environment, which already has them (see README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU deepagents langchain-openai langchain-tavily python-dotenv
# For Tavily search, also add TAVILY_API_KEY to Colab Secrets or your local .env file.

Note: you may need to restart the kernel to use updated packages.


### Load env variables

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 1. The simplest deep agent and its file system
Even without our own tools the agent has the built-in file tools. We ask it to save the result to a file and then read the file from the returned state.

In [3]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:gpt-5-mini",
    system_prompt="You are a helpful assistant. Save every deliverable as a file.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Write a haiku about LangGraph and save it to /haiku.md."}]}
)

# tools the agent decided to call
for message in result["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print("tool call:", call["name"], call["args"].get("file_path", ""))

print("\nFiles in the agent state:", list(result["files"]))
print(result["files"]["/haiku.md"]["content"])

tool call: write_file /haiku.md

Files in the agent state: ['/haiku.md']
Nodes weave whispered paths
Edges hum with bridges of thought
Answers find their home



### 2. Subagents: a researcher and a critic
A subagent is described by a dictionary: `name`, `description` (the main agent uses it to decide when to delegate), `system_prompt` and optional `tools`. The main agent calls a subagent with the built-in `task` tool. The subagent works in its own context and returns only its final answer.

In [4]:
from langchain_tavily import TavilySearch

web_search = TavilySearch(max_results=3)

researcher = {
    "name": "researcher",
    "description": "Searches the web and returns concise, sourced facts on one focused question.",
    "system_prompt": (
        "You are a researcher. Use web search to answer the question you get. "
        "Return 5-8 short bullet points with facts and the source URL for each."
    ),
    "tools": [web_search],
}

critic = {
    "name": "critic",
    "description": "Reviews a draft report saved in a file and lists concrete improvements.",
    "system_prompt": (
        "You are a demanding editor. Read the file you are pointed to and return "
        "at most 5 concrete, actionable improvements. Do not rewrite the report."
    ),
}

analyst = create_deep_agent(
    model="openai:gpt-5-mini",
    subagents=[researcher, critic],
    system_prompt=(
        "You are a lead analyst who writes short research reports.\n"
        "1. Delegate research questions to the `researcher` subagent.\n"
        "2. Write a draft report to /report.md (max 300 words, with a Sources section).\n"
        "3. Ask the `critic` subagent to review /report.md.\n"
        "4. Apply the feedback and save the final version to /report.md.\n"
        "Answer the user with a two-sentence summary."
    ),
)

### 3. Run the agent and watch the delegation
We stream two modes at once: `updates` to print every tool call (subagent delegations and file operations) and `values` to keep the final state.

In [5]:
question = (
    "What are the main differences between LangChain's create_agent and LangGraph's StateGraph, "
    "and when should each be used?"
)

final_state = None
for mode, chunk in analyst.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode=["updates", "values"],
):
    if mode == "values":
        final_state = chunk
        continue
    for node, update in chunk.items():
        messages = (update or {}).get("messages", [])
        messages = getattr(messages, "value", messages)  # some updates wrap the list
        for message in messages:
            for call in getattr(message, "tool_calls", None) or []:
                target = call["args"].get("subagent_type") or call["args"].get("file_path", "")
                print(f"[{node}] {call['name']} -> {target}")

print("\nAgent answer:\n", final_state["messages"][-1].text)

[model] task -> researcher
[model] write_file -> /report.md
[model] task -> critic
[model] write_file -> /report.md

Agent answer:
 LangChain's create_agent is a high-level, batteries-included agent factory that implements the standard model→tool loop (quick to build, middleware-friendly, and LangSmith-ready) with implicit per-run message/state management. LangGraph's StateGraph is an explicit, typed stateful graph runtime for modeling deterministic, checkpointable workflows (nodes, reducers, conditional edges, loops and resume semantics) that gives finer control for complex, long-running orchestration.  

Use create_agent for rapid prototypes and production chatbots or RAG-style assistants where the simple ReAct-style loop suffices; choose StateGraph when you need explicit branching, long-lived sessions, checkpointing/resume, human-in-the-loop approvals, or multi-step deterministic workflows.


### 4. The final report from the virtual file system

In [6]:
from IPython.display import Markdown

Markdown(final_state["files"]["/report.md"]["content"])

Executive summary (short)
Use LangChain's create_agent for fast, short-lived tool-enabled assistants; use LangGraph's StateGraph when you need explicit, checkpointable, multi-step orchestration with deterministic state control.

LangChain create_agent
- Purpose: high-level factory implementing the standard model→tool loop (e.g., ReAct). Best for rapid composition of chatbots and tool-enabled assistants.
- Model & state: implicit loop with per-run message/history state and middleware hooks.
- Strengths: minimal boilerplate, built-in tool integrations, quick LangSmith tracing/observability integration.
- Limitations: less explicit control over branching, long-running flows, and deterministic state transitions — may require additional instrumentation for complex orchestration.

LangGraph StateGraph
- Purpose: explicit stateful graph runtime for modeling workflows as nodes and edges; designed for deterministic, long-running, checkpointable flows.
- Model & state: typed shared State passed between nodes; nodes emit partial updates merged via configurable reducers; edges support conditional routing, loops, and resume/checkpoint semantics.
- Strengths: fine-grained control, clear branching/interrupt patterns, suitable for human-in-the-loop, multi-step workflows, and long-lived sessions.
- Limitations: more boilerplate, steeper learning curve, and additional operational requirements (persistence/backing stores, checkpointing strategy).

Concrete examples
- create_agent: RAG chatbot that calls a search tool and a calculator within a single session.
- StateGraph: multi-step loan application with automated checks, human approval steps, retry logic, and checkpoint/resume across sessions.

Notes & verification
- Verify reducer determinism and exact interrupt semantics in the StateGraph docs for your target version before relying on strict replay guarantees.

Sources (last verified 2026-11-27)
- LangChain Agents docs: https://docs.langchain.com/oss/python/langchain/agents
- create_agent reference: https://reference.langchain.com/python/langchain/agents/factory/create_agent
- LangChain observability: https://docs.langchain.com/oss/python/langchain/observability
- LangGraph overview: https://docs.langchain.com/oss/python/langgraph/overview
- StateGraph API: https://reference.langchain.com/python/langgraph/graph/state/StateGraph
- create_agent example tutorial: https://github.com/microsoft/langchain-for-beginners/blob/main/05-agents/code/01_create_agent_basic.py

### When to use what?
| Need | Tool |
|---|---|
| Model + tools in a loop, short tasks | `langchain.agents.create_agent` |
| Long, multi-step tasks with notes, files and delegation | `deepagents.create_deep_agent` |
| Fully custom control flow (fixed steps, loops, branching) | LangGraph `StateGraph` |

Next steps: give the agent long-term memory (`memory=`), skills (`skills=`), human approval for selected tools (`interrupt_on=`) or a real file system / sandbox through a different `backend`.